In [3]:
import pandas as pd
import numpy as np
import json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

def load_and_clean_data(file_path):
    # 1. Load the dataset
    df = pd.read_csv(file_path)
    
    # 2. Standardize column names (lowercasing for consistency)
    df.columns = [col.lower() for col in df.columns]
    
    # 3. Time formatting
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
    
    # Check if there's any missing values in any of the columns
    for column in df.columns:
        counter = 0
        for item in df[column]:
            if pd.isna(item):
                counter += 1
        if counter==0:
            continue
        print(column, counter) ### no missing values in any of the columns, so we can skip this step. If there were missing values, we would have to decide how to handle them (e.g., fill with mean/median, drop rows, etc.)


    return df

def engineer_features(df):
    # Make a copy to avoid SettingWithCopyWarning
    data = df.copy() 
    
    data['overnight_return'] = data['open'] / data['close'].shift(1) - 1


    # 5. Daily Returns (percentage profits per day)
    data['daily_return'] = data['close'].pct_change() 
    
    # 6. Moving Averages (7-day and 30-day)
    data['sma_7'] = data['close'].rolling(window=7).mean()
    data['sma_30'] = data['close'].rolling(window=30).mean()
    
    # 7. Volatility Index (7-day rolling standard deviation of daily returns)
    data['volatility_7d'] = data['daily_return'].rolling(window=7).std()
    # During times of high volatility, the model might learn that a massive breakout or crash is imminent.

    data['vol_bucket'] = pd.cut(
    data['volatility_7d'],
    bins=[0, 0.06, 0.12, 1],
    labels=['0', '0.5', '1']
    )

    data['risk_adjusted_return'] = data['daily_return'] / (data['volatility_7d'] + 1e-6)


    # 8. 7 Day Price Momentum (Close price compared to 7 days ago)
    # This line tells you how much the token has grown (or shrunk) compared to exactly one week ago.
    data['momentum_7d'] = data['close'] / data['close'].shift(7) - 1
    
    # 2. NEW: Volume Percentage Change
    data['volume_change'] = data['volume'].pct_change()

    # Volume #
    data['volume_ema'] = data['volume'].ewm(span=20, adjust=False).mean()
    data['volume_spike'] = data['volume'] / data['volume_ema']
    data['volume_spike_strength'] = np.tanh(data['volume_spike'] - 1)
    data['is_high_spike'] = (data['volume_spike'] > 2.5).astype(int)



    # 9. Target Variable Generation (1 if price goes UP tomorrow, 0 if DOWN)
    # Where the condition (tomorrow's price > today's price) is true, put a 1. Otherwise, put a 0.
    data['target_up_down'] = np.where(data['close'].shift(-1) > data['close'], 1, 0)
    data['target'] = np.where(data['daily_return'].shift(-1) > 0.025, 1, 0)

    data.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 10. Drop NaN values generated by rolling windows and shifts
    data.dropna(inplace=True)

    return data

# --- Execution Example ---
import os

# 1. Define the list of all your dataset files
all_files = [
   "DATA-INTRA/coin_Tether.csv"
]


all_processed_data = []
all_models_data = {}

# 2. Loop through each file
for file in all_files:
    if os.path.exists(file):
        # Extract the coin's name from the filename
        coin_name = file.replace("coin_", "").replace(".csv", "")
        
        # Run your pipeline
        raw_data = load_and_clean_data(file)
        processed_data = engineer_features(raw_data)
        
        # Add the coin name as a new column so you know which row belongs to which crypto
        processed_data['coin_name'] = coin_name
        
        # --- FEATURES ---
        features = [
            'daily_return','sma_7','sma_30','volatility_7d',
            'risk_adjusted_return','momentum_7d',
            'volume_change','volume_spike_strength','is_high_spike'
        ]

        df=processed_data
        X = df[features]
        y = df['target']

        # 3. SPLIT THE DATA (80% for training, 20% for testing)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 4. Scale the features 
        # CRITICAL: We only 'fit' the scaler on the training data to prevent data leakage,
        # but we 'transform' both the training and testing data.
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        ### ----- Logistic Regression Model ----- ###
        # 5. Train the lightweight classification model on the TRAINING data
        lr_model = LogisticRegression(random_state=42, class_weight='balanced')
        lr_model.fit(X_train_scaled, y_train)

        # 6. TEST THE ACCURACY
        # Make predictions on the 20% unseen test data
        y_pred = lr_model.predict(X_test_scaled)

        # Calculate accuracy metrics
        # Calculate accuracy metrics
        print(f"\n🔥 BEST MODEL for {coin_name}")

        lr_accuracy = accuracy_score(y_test, y_pred)
        print(f"--- Model Performance Logistic Regression ---")
        print(f"Accuracy: {lr_accuracy * 100:.2f}%\n") 
        
        # 7. Extract the parameters (Weights and Bias) needed for frontend inference
        weights = lr_model.coef_[0].tolist()
        bias = float(lr_model.intercept_[0])

        scaler_means = scaler.mean_.tolist()
        scaler_scales = scaler.scale_.tolist()

        # 8. NEW: Save to the master dictionary using the coin's name as the key
        all_models_data[coin_name] = {
            "model_type": "Logistic Regression",
            "accuracy": round(lr_accuracy, 4),
            "features": features,
            "weights": weights,
            "bias": bias,
            "scaler": {
                "means": scaler_means,
                "scales": scaler_scales
            }
        }
        
        # 9. NEW: Export the entire master dictionary to data.json OUTSIDE the loop
        with open("data.json", "w") as f:
            json.dump(all_models_data, f, indent=4)

        print("\nAll models successfully exported to data.json!")



🔥 BEST MODEL for DATA-INTRA/Tether
--- Model Performance Logistic Regression ---
Accuracy: 88.22%


All models successfully exported to data.json!
